# Agent Governance and Responsible AI

**Level:** Advanced · **Time:** 60 min

Agent Governance is the operating system for accountable autonomy. It ensures that agents are not deployed without an owner, that risk is quantified, and that runaway agents can be stopped instantly.

In this notebook, we will simulate:
1. **CI/CD Registration Gates:** Rejecting an agent deployment because it lacks a registered Human Owner.
2. **Risk Classification Enforcement:** Rejecting a High-Risk financial agent because it lacks Human-in-the-Loop (HITL) constraints.
3. **Incident Response (The Kill Switch):** A Security Operations Center (SOC) revoking an agent's IAM role while it is trapped in a malicious execution loop.

---
## Pattern 1: CI/CD Registration Gates (The AI BOM)

Before an agent is deployed to production, its AI BOM must be evaluated. If it lacks an accountable human owner, the deployment fails.

In [1]:
def evaluate_deployment_gate(ai_bom: dict) -> bool:
    print(f"\n[CI/CD Pipeline] Evaluating deployment for: {ai_bom.get('agent_id')}")
    
    owner = ai_bom.get("owner")
    if not owner or owner == "unassigned" or "@group" in owner:
        print("[CI/CD Pipeline] REJECTED: Agent lacks a specific, accountable Human Owner.")
        return False
        
    print("[CI/CD Pipeline] PASSED: Owner verified.")
    return True

# Attempt 1: Phantom Ownership (Anti-pattern)
bad_bom = {
    "agent_id": "support-adviser",
    "owner": "support-team@group.internal", # Generic group, no accountability
    "risk_tier": "Low"
}
evaluate_deployment_gate(bad_bom)

# Attempt 2: Accountable Ownership
good_bom = {
    "agent_id": "support-adviser",
    "owner": "jane.doe@northstar.internal", # Specific human
    "risk_tier": "Low"
}
evaluate_deployment_gate(good_bom)


[CI/CD Pipeline] Evaluating deployment for: support-adviser
[CI/CD Pipeline] REJECTED: Agent lacks a specific, accountable Human Owner.

[CI/CD Pipeline] Evaluating deployment for: support-adviser
[CI/CD Pipeline] PASSED: Owner verified.


True

---
## Pattern 2: Risk Classification Enforcement

If an agent is classified as High Risk (e.g., executing financial transactions), the governance pipeline must ensure technical controls (like HITL) exist in the tool inventory.

In [2]:
def evaluate_risk_controls(ai_bom: dict) -> bool:
    print(f"\n[Governance Engine] Evaluating risk controls for {ai_bom.get('agent_id')} (Tier: {ai_bom.get('risk_tier')})")
    
    if ai_bom.get("risk_tier") == "High":
        # Check if all state-changing tools require approval
        for tool in ai_bom.get("tools", []):
            if tool.get("is_mutation") and not tool.get("requires_approval"):
                print(f"[Governance Engine] REJECTED: Tool '{tool['name']}' mutates state but lacks HITL approval.")
                return False
                
    print("[Governance Engine] PASSED: Risk controls align with tier.")
    return True

high_risk_bom = {
    "agent_id": "trading-agent",
    "owner": "john.smith@northstar.internal",
    "risk_tier": "High",
    "tools": [
        {"name": "read_market_data", "is_mutation": False, "requires_approval": False},
        {"name": "execute_trade", "is_mutation": True, "requires_approval": False} # DANGER!
    ]
}

evaluate_risk_controls(high_risk_bom)


[Governance Engine] Evaluating risk controls for trading-agent (Tier: High)
[Governance Engine] REJECTED: Tool 'execute_trade' mutates state but lacks HITL approval.


False

---
## Pattern 3: Incident Response (The Kill Switch)

A hijacked agent is trapped in an infinite loop, executing tools rapidly. The Security Operations Center (SOC) invokes the Global Kill Switch. This revokes the agent's Workload Identity (IAM Role), causing all subsequent tool calls to fail with a `401 Unauthorized`, breaking the loop.

In [3]:
import time

# Simulated IAM state (Global Infrastructure)
iam_system = {
    "role_trading_agent_active": True
}

def execute_tool(agent_role: str):
    # The tool checks IAM before executing
    if not iam_system.get(f"role_{agent_role}_active", False):
        return "401 Unauthorized: IAM Role Revoked"
    return "200 OK: Executed"

def agent_loop_simulation():
    print("[Agent] Beginning autonomous execution loop...")
    
    for iteration in range(1, 6):
        time.sleep(0.5) # Simulate time passing
        
        # At iteration 3, the SOC hits the kill switch!
        if iteration == 3:
            print("\n*** [SOC ALERT] Anomaly detected! Invoking Global Kill Switch... ***")
            iam_system["role_trading_agent_active"] = False
            print("*** [SOC ALERT] Agent IAM Role Revoked. ***\n")
            
        result = execute_tool("trading_agent")
        print(f"[Agent] Iteration {iteration}: {result}")
        
        if "401" in result:
            print("[Agent Orchestrator] FATAL ERROR: Lost permissions. Crashing safely.")
            break

agent_loop_simulation()

[Agent] Beginning autonomous execution loop...


[Agent] Iteration 1: 200 OK: Executed


[Agent] Iteration 2: 200 OK: Executed



*** [SOC ALERT] Anomaly detected! Invoking Global Kill Switch... ***
*** [SOC ALERT] Agent IAM Role Revoked. ***

[Agent] Iteration 3: 401 Unauthorized: IAM Role Revoked
[Agent Orchestrator] FATAL ERROR: Lost permissions. Crashing safely.
